# Upper-Body Clothing Detection with YOLOv8
Detects 6 upper-body clothing categories from DeepFashion2:
- Short sleeve top
- Long sleeve top  
- Short sleeve outwear
- Long sleeve outwear
- Vest
- Sling

In [ ]:
# Install dependencies (run once)
!pip install ultralytics opencv-python -q

In [ ]:
# Imports
import cv2
import gc
from ultralytics import YOLO

In [ ]:
# =============================================================================
# CLASS CONFIGURATION - DeepFashion2 Upper-Body Categories
# =============================================================================
# These are the 6 upper-body clothing classes we want to detect.
# Modify this dictionary if your model uses different class indices.
# Format: {class_index: "class_name"}

TARGET_CLASSES = {
    0: "short sleeve top",
    1: "long sleeve top",
    2: "short sleeve outwear",
    3: "long sleeve outwear",
    4: "vest",
    5: "sling"
}

# List of class indices to filter (only detect these)
TARGET_CLASS_IDS = list(TARGET_CLASSES.keys())

# Colors for each class (BGR format)
CLASS_COLORS = {
    0: (255, 100, 100),   # Light blue - short sleeve top
    1: (255, 0, 0),       # Blue - long sleeve top
    2: (100, 255, 100),   # Light green - short sleeve outwear
    3: (0, 255, 0),       # Green - long sleeve outwear
    4: (100, 100, 255),   # Light red - vest
    5: (0, 255, 255)      # Yellow - sling
}

print("✅ Target classes configured:")
for idx, name in TARGET_CLASSES.items():
    print(f"   Class {idx}: {name}")

In [ ]:
# =============================================================================
# LOAD YOLOV8 MODEL
# =============================================================================
# Option 1: Load custom trained weights (DeepFashion2)
# model = YOLO("path/to/your/deepfashion2_weights.pt")

# Option 2: For testing, use pretrained YOLOv8n (will detect COCO classes)
# Replace with your custom weights path when available
MODEL_PATH = "yolov8n.pt"  # <-- CHANGE THIS to your custom weights

model = YOLO(MODEL_PATH)
print(f"✅ Model loaded: {MODEL_PATH}")
print(f"   Model classes: {model.names}")

In [ ]:
# =============================================================================
# MAIN DETECTION FUNCTION
# =============================================================================

def detect_upper_body_clothes(source=0, conf_threshold=0.5, frame_scale=0.5):
    """
    Real-time upper-body clothing detection using YOLOv8.
    
    Args:
        source: 0 for webcam, or path to video file
        conf_threshold: Minimum confidence for detections (0.0 to 1.0)
        frame_scale: Scale factor for processing (lower = faster, less accurate)
    """
    cap = cv2.VideoCapture(source)
    
    if not cap.isOpened():
        print("❌ Cannot open video source")
        return
    
    # Get original frame dimensions
    orig_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"🎥 Video source: {orig_width}x{orig_height}")
    print(f"🔧 Processing scale: {frame_scale} ({int(orig_width*frame_scale)}x{int(orig_height*frame_scale)})")
    print("Press 'q' to quit\n")
    
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # ---------------------------------------------------------------------
        # FRAME DOWNSCALING FOR EFFICIENCY
        # ---------------------------------------------------------------------
        # Resize frame for faster inference (optional)
        if frame_scale != 1.0:
            proc_frame = cv2.resize(frame, None, fx=frame_scale, fy=frame_scale)
        else:
            proc_frame = frame.copy()
        
        # ---------------------------------------------------------------------
        # YOLOV8 INFERENCE
        # ---------------------------------------------------------------------
        results = model.predict(
            proc_frame,
            conf=conf_threshold,
            imgsz=320,           # Small input size for speed
            max_det=20,          # Max detections per frame
            verbose=False        # Suppress console output
        )
        
        # ---------------------------------------------------------------------
        # CLASS FILTERING - Only process TARGET_CLASSES
        # ---------------------------------------------------------------------
        # This is where we filter detections to only our 6 clothing categories
        
        detections = results[0].boxes
        detection_count = 0
        
        for box in detections:
            # Get class ID and confidence
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            
            # =================================================================
            # CLASS FILTER: Skip classes not in TARGET_CLASS_IDS
            # To change which classes are detected, modify TARGET_CLASSES dict
            # =================================================================
            if class_id not in TARGET_CLASS_IDS:
                continue  # Skip this detection
            
            detection_count += 1
            
            # Get bounding box coordinates (scaled back to original frame)
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if frame_scale != 1.0:
                x1 = int(x1 / frame_scale)
                y1 = int(y1 / frame_scale)
                x2 = int(x2 / frame_scale)
                y2 = int(y2 / frame_scale)
            
            # Get class name and color
            class_name = TARGET_CLASSES.get(class_id, f"Class {class_id}")
            color = CLASS_COLORS.get(class_id, (255, 255, 255))
            
            # -----------------------------------------------------------------
            # DRAW BOUNDING BOX
            # -----------------------------------------------------------------
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # -----------------------------------------------------------------
            # DRAW LABEL WITH CONFIDENCE
            # -----------------------------------------------------------------
            label = f"{class_name}: {confidence:.2f}"
            
            # Label background
            (label_w, label_h), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
            )
            cv2.rectangle(
                frame, 
                (x1, y1 - label_h - 10), 
                (x1 + label_w + 5, y1), 
                color, 
                -1  # Filled
            )
            
            # Label text
            cv2.putText(
                frame, 
                label, 
                (x1 + 2, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 
                0.6, 
                (255, 255, 255),  # White text
                2
            )
        
        # ---------------------------------------------------------------------
        # DISPLAY STATUS OVERLAY
        # ---------------------------------------------------------------------
        status = f"Detections: {detection_count} | Frame: {frame_count}"
        cv2.rectangle(frame, (10, 10), (350, 40), (0, 0, 0), -1)
        cv2.putText(frame, status, (15, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Show frame
        cv2.imshow("Upper-Body Clothing Detection", frame)
        
        # Quit on 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        # Memory cleanup every 100 frames
        if frame_count % 100 == 0:
            gc.collect()
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"✅ Detection stopped. Total frames: {frame_count}")

In [ ]:
# =============================================================================
# RUN DETECTION
# =============================================================================
# source=0        -> Webcam
# source="video.mp4" -> Video file
# conf_threshold  -> Minimum confidence (0.0-1.0)
# frame_scale     -> Processing scale (0.5 = half size, faster)

detect_upper_body_clothes(
    source=0,              # Webcam
    conf_threshold=0.5,    # 50% confidence threshold
    frame_scale=0.5        # Process at half resolution for speed
)

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================
del model
gc.collect()
print("✅ Cleanup complete")

# Google Colab Version
Since Colab doesn't support `cv2.imshow()`, use this version to process uploaded videos and display frames inline.

In [ ]:
# =============================================================================
# GOOGLE COLAB - FULL CODE (Copy everything below to Colab)
# =============================================================================

# Cell 1: Install dependencies
!pip install ultralytics opencv-python -q

# Cell 2: Imports and Configuration
import cv2
import gc
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow
from IPython.display import display, clear_output, HTML
from ultralytics import YOLO
import time

# =============================================================================
# CLASS CONFIGURATION - DeepFashion2 Upper-Body Categories
# =============================================================================
TARGET_CLASSES = {
    0: "short sleeve top",
    1: "long sleeve top",
    2: "short sleeve outwear",
    3: "long sleeve outwear",
    4: "vest",
    5: "sling"
}

TARGET_CLASS_IDS = list(TARGET_CLASSES.keys())

CLASS_COLORS = {
    0: (255, 100, 100),   # Light blue - short sleeve top
    1: (255, 0, 0),       # Blue - long sleeve top
    2: (100, 255, 100),   # Light green - short sleeve outwear
    3: (0, 255, 0),       # Green - long sleeve outwear
    4: (100, 100, 255),   # Light red - vest
    5: (0, 255, 255)      # Yellow - sling
}

print("✅ Configuration loaded")
print(f"Target classes: {list(TARGET_CLASSES.values())}")

# Cell 3: Load Model
MODEL_PATH = "yolov8n.pt"  # Change to your custom weights
model = YOLO(MODEL_PATH)
print(f"✅ Model loaded: {MODEL_PATH}")

# Cell 4: Upload Video
print("📁 Upload a video file:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"✅ Video uploaded: {video_path}")

# Cell 5: Process Video and Display Results
def process_video_colab(video_path, conf_threshold=0.5, frame_scale=0.5, 
                        max_frames=200, display_every=5):
    """
    Process video for Google Colab with inline display.
    
    Args:
        video_path: Path to uploaded video
        conf_threshold: Minimum confidence (0.0-1.0)
        frame_scale: Processing scale (0.5 = half size)
        max_frames: Maximum frames to process
        display_every: Display every Nth frame (reduces output)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("❌ Cannot open video")
        return
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"🎥 Video: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames")
    print(f"🔧 Processing {min(max_frames, total_frames)} frames...")
    print("-" * 50)
    
    frame_count = 0
    start_time = time.time()
    
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Downscale for processing
        if frame_scale != 1.0:
            proc_frame = cv2.resize(frame, None, fx=frame_scale, fy=frame_scale)
        else:
            proc_frame = frame.copy()
        
        # YOLOv8 inference
        results = model.predict(
            proc_frame,
            conf=conf_threshold,
            imgsz=320,
            max_det=20,
            verbose=False
        )
        
        # Process detections
        detections = results[0].boxes
        detection_count = 0
        
        for box in detections:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            
            # Filter to target classes only
            if class_id not in TARGET_CLASS_IDS:
                continue
            
            detection_count += 1
            
            # Scale coordinates back
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            if frame_scale != 1.0:
                x1 = int(x1 / frame_scale)
                y1 = int(y1 / frame_scale)
                x2 = int(x2 / frame_scale)
                y2 = int(y2 / frame_scale)
            
            class_name = TARGET_CLASSES.get(class_id, f"Class {class_id}")
            color = CLASS_COLORS.get(class_id, (255, 255, 255))
            
            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Draw label
            label = f"{class_name}: {confidence:.2f}"
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1-lh-10), (x1+lw+5, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Status overlay
        status = f"Frame {frame_count}/{min(max_frames, total_frames)} | Detections: {detection_count}"
        cv2.rectangle(frame, (10, 10), (400, 40), (0, 0, 0), -1)
        cv2.putText(frame, status, (15, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        # Display every Nth frame
        if frame_count % display_every == 0:
            clear_output(wait=True)
            # Convert BGR to RGB for display
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize for display if too large
            if frame.shape[1] > 800:
                scale = 800 / frame.shape[1]
                frame_rgb = cv2.resize(frame_rgb, None, fx=scale, fy=scale)
            cv2_imshow(frame_rgb)
    
    cap.release()
    
    elapsed = time.time() - start_time
    print(f"\n✅ Processing complete!")
    print(f"   Frames: {frame_count}")
    print(f"   Time: {elapsed:.1f}s")
    print(f"   FPS: {frame_count/elapsed:.1f}")
    
    gc.collect()

# Cell 6: Run Processing
process_video_colab(
    video_path,
    conf_threshold=0.5,
    frame_scale=0.5,
    max_frames=200,      # Limit frames for Colab
    display_every=10     # Show every 10th frame
)

# Cell 7: Alternative - Use Webcam in Colab (requires browser permission)
# from google.colab import output
# from IPython.display import Javascript
# 
# def use_webcam_colab():
#     js = Javascript('''
#         async function takePhoto() {
#             const stream = await navigator.mediaDevices.getUserMedia({video: true});
#             const video = document.createElement('video');
#             document.body.appendChild(video);
#             video.srcObject = stream;
#             await video.play();
#             
#             const canvas = document.createElement('canvas');
#             canvas.width = video.videoWidth;
#             canvas.height = video.videoHeight;
#             canvas.getContext('2d').drawImage(video, 0, 0);
#             
#             stream.getTracks().forEach(track => track.stop());
#             video.remove();
#             return canvas.toDataURL('image/jpeg');
#         }
#     ''')
#     display(js)
#     # Use: data = output.eval_js('takePhoto()')

# Cell 8: Cleanup
del model
gc.collect()
print("✅ Cleanup complete")